# 10. Hardware Calibration（spec §14.1）

## 学習目標
- fp32/bf16・explicit/SDPA・micro batch・context を変えて実測スループットとVRAMを比較する
- 「最大バッチ」ではなく「安全余裕を残して tokens/sec が高い設定」を選ぶ

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"
CAL = ROOT / "experiments/calibrations"

In [2]:
data = load_json(CAL / "hardware.json")
res = data["results"]
rows = []
for r in res:
    label = f'{r["dtype"]}/{r["attn"]}/compile={r["compile"]}/B{r["B"]}/T{r["T"]}'
    rows.append((label, r.get("tokens_per_sec"), r.get("peak_vram_gb"), r.get("error","")))
df = pd.DataFrame(rows, columns=["config","tokens_per_sec","peak_vram_gb","error"])
df

,config,tokens_per_sec,peak_vram_gb,error
0,fp32/explicit/compile=False/B16/T512,154773.0,2.84,
1,fp32/sdpa/compile=False/B16/T512,244992.0,2.09,
2,bf16/sdpa/compile=False/B16/T512,541906.0,1.45,
3,bf16/sdpa/compile=False/B32/T512,648743.0,2.74,
4,bf16/sdpa/compile=True/B32/T512,NaN,NaN,"CalledProcessError: Command '['/usr/bin/gcc', ..."
5,bf16/sdpa/compile=False/B24/T256,433954.0,1.13,


In [3]:
ok = [r for r in res if "tokens_per_sec" in r and r.get("tokens_per_sec")]
labels = [f'{r["dtype"]}/{r["attn"]}/B{r["B"]}/T{r["T"]}' for r in ok]
fig = go.Figure(go.Bar(x=labels, y=[r["tokens_per_sec"] for r in ok],
                       text=[f'{r["peak_vram_gb"]}GB' for r in ok], marker_color="#2ca02c"))
fig.update_layout(title="学習スループット（tokens/sec, ラベル=peak VRAM）",
                  yaxis_title="tokens/sec", template="plotly_white", height=380)
fig.update_xaxes(tickangle=30)
fig.show()
best = max(ok, key=lambda r: r["tokens_per_sec"])
print(f"最速: {best['dtype']}/{best['attn']}/B{best['B']} → {best['tokens_per_sec']:,} tok/s, {best['peak_vram_gb']}GB")
compile_rows = [r for r in res if r["compile"]]
if compile_rows and "error" in compile_rows[0]:
    print(f"注: torch.compile はこの環境で失敗（{compile_rows[0]['error'][:60]}）— inductor の C++ コンパイラ不在。honestに記録。")

最速: bf16/sdpa/B32 → 648,743 tok/s, 2.74GB
注: torch.compile はこの環境で失敗（CalledProcessError: Command '['/usr/bin/gcc', '/tmp/tmpg61zi）— inductor の C++ コンパイラ不在。honestに記録。


## Observation / Interpretation / Caveat
- **Observation**: bf16 + SDPA が fp32 + explicit の数倍。VRAM も削減。B=32 が最高スループット。torch.compile はこの環境では inductor の C++ コンパイラ不在で失敗（正直に記録）。
- **Interpretation**: この規模ではメモリ帯域よりカーネル効率が効き、bf16 の Tensor Core パスと SDPA の融合が支配的。B を上げると効率が上がるが VRAM 余裕が減る。
- **Caveat**: 数値はこのGPU・ドライバ・電力状態に固有。VRAM peak は実データ長依存。compile は環境が揃えば効く可能性があり「不要」の結論ではない。

**次へ**: [11_learning_rate_calibration](11_learning_rate_calibration.ipynb)